In [1]:
# Fine-tune RoBERTa-large-MNLI on two-class FEVER (evidence = premise, claim = hypothesis), keeping its three-way head.
# TASK_B_SMOKE_TEST=1 (or SMOKE_TEST = True) runs only the first smoke steps: an engineering check, never a result.
import hashlib, json, math, os, sys, time
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup, set_seed)

PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "src/task_b").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the NLP_PROJECT repository folder in VS Code before running this notebook")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from task_b.fever import claim_key
from task_b.models import joined_premise, model_fingerprint, nli_joined_predictions, nli_label_ids, nli_predictor
from task_b.schema import parse_claim_evidence
from task_b.verification import evaluate, fever_binary_proxy, new_run_dir, save_run, timed_predictions

config = json.loads((PROJECT_ROOT / "configs/task_b_fever_finetune_two_class.json").read_text(encoding="utf-8"))
SMOKE_TEST = os.environ.get("TASK_B_SMOKE_TEST") == "1"
MODE = "smoke test (engineering check, not an evaluation)" if SMOKE_TEST else "full training run"
LABELS, SEED = ("SUPPORTED", "CONTRADICTED"), config["seed"]
set_seed(SEED)  # seeds Python, NumPy and PyTorch
print(MODE.upper())

c:\Users\ONYX\Desktop\Aadit\N025\prac\NLP\NLP_PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FULL TRAINING RUN


In [2]:
# Training split (checked against its build manifest), validation and test records; gold labels and evidence unchanged.
def read_records(path):
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return rows, [parse_claim_evidence(row["claim"]) for row in rows], [row["gold_label"] for row in rows]

split_dir = PROJECT_ROOT / config["train_split_dir"]
split_manifest = json.loads((split_dir / "split_manifest.json").read_text(encoding="utf-8"))
assert hashlib.sha256((split_dir / "records.jsonl").read_bytes()).hexdigest() == split_manifest["outputs"]["records.jsonl"], \
    "training split changed since it was built"
train_rows, train_examples, train_gold = read_records(split_dir / "records.jsonl")
validation_rows, validation_examples, validation_gold = read_records(PROJECT_ROOT / config["validation_dir"] / "records.jsonl")
test_rows, test_examples, test_gold = read_records(PROJECT_ROOT / config["test_dir"] / "records.jsonl")

In [3]:
# Leakage checks: no training claim text (case and spacing ignored) or id in validation or test; same test ids as zero-shot.
train_keys = {claim_key(example.claim) for example in train_examples}
train_ids = {str(row["raw_id"]) for row in train_rows}
for name, rows, examples in [("validation", validation_rows, validation_examples), ("test", test_rows, test_examples)]:
    assert not train_keys & {claim_key(example.claim) for example in examples}, f"a training claim text is in {name}"
    assert not train_ids & {str(row["raw_id"]) for row in rows}, f"a training id is in {name}"
assert Counter(train_gold) == {"SUPPORTED": len(train_gold) // 2, "CONTRADICTED": len(train_gold) // 2}
zero_shot_rows = [json.loads(line) for line in (PROJECT_ROOT / config["zero_shot_test_run"] / "predictions.jsonl")
                  .read_text(encoding="utf-8").splitlines() if line.strip()]
assert [row["claim_id"] for row in zero_shot_rows] == [row["claim"]["claim_id"] for row in test_rows], \
    "test claims differ from the zero-shot run"
pd.DataFrame({name: Counter(gold) for name, gold in [("train", train_gold), ("validation", validation_gold), ("test", test_gold)]}).T

,SUPPORTED,CONTRADICTED
train,26689,26689
validation,1978,1987
test,6597,6605


In [4]:
# Local RoBERTa-large-MNLI snapshot (never downloaded); class ids are read by name from its config, never assumed.
assert torch.cuda.is_available(), "a CUDA GPU is required to fine-tune RoBERTa-large"
device = torch.device("cuda")
transformers.logging.set_verbosity_error()  # hide the routine report about the unused pooler weights
MODEL_DIR = PROJECT_ROOT / config["model_dir"]
assert model_fingerprint(MODEL_DIR) == config["model_snapshot_sha256"], "local snapshot differs from the zero-shot one"
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR), local_files_only=True).to(device)
BASE_ID2LABEL, ids = dict(model.config.id2label), nli_label_ids(model.config)
TARGET_IDS = {label: ids[name] for label, name in config["label_targets"].items()}
assert TARGET_IDS == config["expected_label_ids"], f"unexpected label ids {TARGET_IDS}"
model.gradient_checkpointing_enable()  # needed to fit an 8 GB GPU
use_bf16 = config["bf16_autocast"] and torch.cuda.is_bf16_supported()
print(torch.cuda.get_device_name(0), "| label ids", TARGET_IDS, "| bf16", use_bf16)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8839.82it/s]


NVIDIA GeForce RTX 4060 Laptop GPU | label ids {'SUPPORTED': 2, 'CONTRADICTED': 0} | bf16 True


In [5]:
# Training pairs: joined evidence as premise, claim as hypothesis; only the evidence is truncated, padding is per batch.
MAX_LEN = config["max_length"]
encoded = tokenizer([joined_premise(example) for example in train_examples], [example.claim for example in train_examples],
                    truncation="only_first", max_length=MAX_LEN)
train_data = [{"input_ids": input_ids, "attention_mask": mask, "labels": TARGET_IDS[gold]}
              for input_ids, mask, gold in zip(encoded["input_ids"], encoded["attention_mask"], train_gold)]
truncated_share = sum(len(row["input_ids"]) == MAX_LEN for row in train_data) / len(train_data)
print(f"{len(train_data):,} training pairs; {truncated_share:.2%} truncated to {MAX_LEN} tokens")

53,378 training pairs; 0.28% truncated to 256 tokens


In [6]:
# Scoring under both protocols in fp32 eval mode, as in the zero-shot run. Primary: the unchanged per-sentence nli_predictor.
def score(examples, gold):
    model.eval()
    start = time.perf_counter()
    per_sentence, latency = timed_predictions(examples, nli_predictor(tokenizer, model, torch))
    middle = time.perf_counter()
    joined, probabilities = nli_joined_predictions(tokenizer, model, torch, examples, MAX_LEN, config["eval_batch_size"])
    result = {"seconds": {"per_sentence": middle - start, "joined": time.perf_counter() - middle}}
    for protocol, raw in [("per_sentence", per_sentence), ("joined", joined)]:
        assert len(raw) == len(gold) and set(raw) <= {"SUPPORTED", "CONTRADICTED", "UNSUPPORTED"}
        binary = [fever_binary_proxy(label) for label in raw]
        metrics = evaluate(gold, binary, LABELS)
        metrics["raw_unsupported_count"] = raw.count("UNSUPPORTED")
        result[protocol] = {"metrics": metrics, "raw": raw, "binary": binary}
    result["joined"]["probabilities"] = probabilities
    model.train()
    return result

collate = DataCollatorWithPadding(tokenizer)
probe_batch = tokenizer([joined_premise(e) for e in validation_examples[:32]], [e.claim for e in validation_examples[:32]],
                        truncation="only_first", max_length=MAX_LEN, padding=True, return_tensors="pt")

@torch.no_grad()
def probe_logits(current_model):
    """Fixed validation pairs in fp32 eval mode, used to confirm a saved checkpoint reloads exactly."""
    current_model.eval()
    logits = current_model(**{key: value.to(device) for key, value in probe_batch.items()}).logits.float().cpu()
    current_model.train()
    return logits

In [7]:
# Seeded shuffled batches; AdamW with linear warm-up and decay over the full schedule (a smoke run follows its first steps).
train_loader = DataLoader(train_data, batch_size=config["train_batch_size"], shuffle=True, collate_fn=collate,
                          generator=torch.Generator().manual_seed(SEED))
ACCUMULATION = config["gradient_accumulation_steps"]
total_steps = math.ceil(len(train_loader) / ACCUMULATION) * config["epochs"]
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
scheduler = get_linear_schedule_with_warmup(optimizer, int(config["warmup_ratio"] * total_steps), total_steps)
max_steps = min(total_steps, config["smoke"]["max_optimizer_steps"]) if SMOKE_TEST else total_steps
n_evals = config["evaluations_per_epoch"] * config["epochs"]
eval_steps = sorted(s for s in (config["smoke"]["eval_at_steps"] if SMOKE_TEST else
                                {round(total_steps * k / n_evals) for k in range(1, n_evals + 1)}) if s <= max_steps)
RUN_DIR = new_run_dir(PROJECT_ROOT / config["output_dir"], "smoke_run_" if SMOKE_TEST else "run_")
MODEL_OUT = RUN_DIR / "model"
print(f"{total_steps:,} optimizer steps per full schedule (warm-up {int(config['warmup_ratio'] * total_steps)}); "
      f"running {max_steps:,}; validation at steps {eval_steps}; run folder {RUN_DIR.relative_to(PROJECT_ROOT)}")

1,669 optimizer steps per full schedule (warm-up 100); running 1,669; validation at steps [417, 834, 1252, 1669]; run folder outputs\task_b\fever_finetune\run_20261022T191303Z


In [8]:
# Step 0: the unchanged zero-shot model on validation, the reference that fine-tuned checkpoints are compared with.
zero_shot_validation = score(validation_examples, validation_gold)
print({protocol: round(zero_shot_validation[protocol]["metrics"]["macro_f1"], 4) for protocol in ("per_sentence", "joined")},
      {key: round(value, 1) for key, value in zero_shot_validation["seconds"].items()}, "seconds")

{'per_sentence': 0.8066, 'joined': 0.8294} {'per_sentence': 64.8, 'joined': 77.6} seconds


In [9]:
# Train on the training split only. At each evaluation step score validation and save the checkpoint with the best
# per-sentence (primary) macro-F1, ties keeping the earlier step. The test set is not used here.
history, loss_log, window = [], [], []
best = {"step": None, "macro_f1": -1.0}
step, examples_seen, eval_seconds = 0, 0, 0.0
torch.cuda.reset_peak_memory_stats()
model.train()
start = time.perf_counter()
for epoch in range(config["epochs"]):
    for batch_index, batch in enumerate(train_loader):
        batch = {key: value.to(device) for key, value in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(**batch).logits
        loss = torch.nn.functional.cross_entropy(logits.float(), labels)
        (loss / ACCUMULATION).backward()
        window.append(loss.item())
        examples_seen += len(labels)
        if (batch_index + 1) % ACCUMULATION and batch_index + 1 < len(train_loader):
            continue
        torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        step += 1
        if step % config["loss_log_every_steps"] == 0:
            loss_log.append({"step": step, "mean_loss": sum(window) / len(window), "lr": scheduler.get_last_lr()[0],
                             "elapsed_seconds": time.perf_counter() - start})
            window = []
            print(f"step {step:>5}: loss {loss_log[-1]['mean_loss']:.4f}  lr {loss_log[-1]['lr']:.2e}  "
                  f"{loss_log[-1]['elapsed_seconds']:.0f}s")
        if step in eval_steps:
            pause = time.perf_counter()
            result = score(validation_examples, validation_gold)
            primary = result["per_sentence"]["metrics"]["macro_f1"]
            history.append({"step": step, "examples_seen": examples_seen, "per_sentence_macro_f1": primary,
                            "joined_macro_f1": result["joined"]["metrics"]["macro_f1"],
                            "per_sentence_raw_unsupported": result["per_sentence"]["metrics"]["raw_unsupported_count"],
                            "eval_seconds": sum(result["seconds"].values())})
            if primary > best["macro_f1"]:
                best = {"step": step, "macro_f1": primary, "validation": result}
                model.save_pretrained(MODEL_OUT)  # weights (safetensors) and config with the original id2label
                tokenizer.save_pretrained(MODEL_OUT)
                saved_probe = probe_logits(model)
            history[-1]["selected_so_far"] = best["step"]
            eval_seconds += time.perf_counter() - pause
            print(f"step {step:>5}: validation macro-F1 per-sentence {primary:.4f} | joined "
                  f"{history[-1]['joined_macro_f1']:.4f} | best step {best['step']}")
        if step >= max_steps:
            break
    if step >= max_steps:
        break
torch.cuda.synchronize()
wall_seconds = time.perf_counter() - start
train_seconds = wall_seconds - eval_seconds
memory = {"peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30, "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30}
timing = {"optimizer_steps": step, "examples_seen": examples_seen, "wall_seconds": wall_seconds, "train_seconds": train_seconds,
          "eval_and_save_seconds": eval_seconds, "train_examples_per_second": examples_seen / train_seconds,
          "seconds_per_optimizer_step": train_seconds / step}
assert best["step"] is not None, "no validation step ran, so no checkpoint was selected"
print(timing, memory)

step    10: loss 0.3791  lr 1.00e-06  8s
step    20: loss 0.3509  lr 2.00e-06  16s
step    30: loss 0.3067  lr 3.00e-06  24s
step    40: loss 0.2645  lr 4.00e-06  32s
step    50: loss 0.1959  lr 5.00e-06  40s
step    60: loss 0.1605  lr 6.00e-06  48s
step    70: loss 0.1472  lr 7.00e-06  56s
step    80: loss 0.1566  lr 8.00e-06  64s
step    90: loss 0.1601  lr 9.00e-06  71s
step   100: loss 0.1106  lr 1.00e-05  79s
step   110: loss 0.1686  lr 9.94e-06  87s
step   120: loss 0.1939  lr 9.87e-06  94s
step   130: loss 0.1848  lr 9.81e-06  102s
step   140: loss 0.1920  lr 9.75e-06  110s
step   150: loss 0.1768  lr 9.68e-06  118s
step   160: loss 0.2014  lr 9.62e-06  126s
step   170: loss 0.1425  lr 9.55e-06  133s
step   180: loss 0.1315  lr 9.49e-06  141s
step   190: loss 0.1643  lr 9.43e-06  149s
step   200: loss 0.1252  lr 9.36e-06  157s
step   210: loss 0.1702  lr 9.30e-06  164s
step   220: loss 0.2852  lr 9.24e-06  173s
step   230: loss 0.1819  lr 9.17e-06  181s
step   240: loss 0.1311 

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.04s/it]


step   417: validation macro-F1 per-sentence 0.9541 | joined 0.9584 | best step 417
step   420: loss 0.1436  lr 7.96e-06  687s
step   430: loss 0.1653  lr 7.90e-06  703s
step   440: loss 0.2532  lr 7.83e-06  718s
step   450: loss 0.1602  lr 7.77e-06  736s
step   460: loss 0.1880  lr 7.71e-06  752s
step   470: loss 0.1869  lr 7.64e-06  770s
step   480: loss 0.1238  lr 7.58e-06  785s
step   490: loss 0.2083  lr 7.51e-06  804s
step   500: loss 0.1189  lr 7.45e-06  820s
step   510: loss 0.1532  lr 7.39e-06  836s
step   520: loss 0.1457  lr 7.32e-06  851s
step   530: loss 0.1931  lr 7.26e-06  868s
step   540: loss 0.1613  lr 7.20e-06  884s
step   550: loss 0.1854  lr 7.13e-06  899s
step   560: loss 0.0865  lr 7.07e-06  916s
step   570: loss 0.1330  lr 7.00e-06  933s
step   580: loss 0.1391  lr 6.94e-06  949s
step   590: loss 0.1269  lr 6.88e-06  965s
step   600: loss 0.1764  lr 6.81e-06  984s
step   610: loss 0.2208  lr 6.75e-06  999s
step   620: loss 0.1153  lr 6.69e-06  1014s
step   630: 

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.31s/it]


step   834: validation macro-F1 per-sentence 0.9571 | joined 0.9624 | best step 834
step   840: loss 0.1293  lr 5.28e-06  1811s
step   850: loss 0.0556  lr 5.22e-06  1826s
step   860: loss 0.1656  lr 5.16e-06  1839s
step   870: loss 0.1448  lr 5.09e-06  1851s
step   880: loss 0.1845  lr 5.03e-06  1864s
step   890: loss 0.1131  lr 4.96e-06  1882s
step   900: loss 0.1264  lr 4.90e-06  1898s
step   910: loss 0.1231  lr 4.84e-06  1916s
step   920: loss 0.1556  lr 4.77e-06  1933s
step   930: loss 0.1553  lr 4.71e-06  1950s
step   940: loss 0.1205  lr 4.65e-06  1963s
step   950: loss 0.1693  lr 4.58e-06  1977s
step   960: loss 0.1578  lr 4.52e-06  1994s
step   970: loss 0.1485  lr 4.46e-06  2009s
step   980: loss 0.1626  lr 4.39e-06  2026s
step   990: loss 0.2052  lr 4.33e-06  2046s
step  1000: loss 0.1349  lr 4.26e-06  2062s
step  1010: loss 0.1385  lr 4.20e-06  2081s
step  1020: loss 0.1151  lr 4.14e-06  2093s
step  1030: loss 0.0717  lr 4.07e-06  2106s
step  1040: loss 0.1393  lr 4.01e-06

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.03s/it]


step  1252: validation macro-F1 per-sentence 0.9607 | joined 0.9662 | best step 1252
step  1260: loss 0.1075  lr 2.61e-06  2822s
step  1270: loss 0.1608  lr 2.54e-06  2839s
step  1280: loss 0.1284  lr 2.48e-06  2852s
step  1290: loss 0.1574  lr 2.42e-06  2867s
step  1300: loss 0.1179  lr 2.35e-06  2879s
step  1310: loss 0.0613  lr 2.29e-06  2893s
step  1320: loss 0.1506  lr 2.22e-06  2906s
step  1330: loss 0.1108  lr 2.16e-06  2919s
step  1340: loss 0.1088  lr 2.10e-06  2934s
step  1350: loss 0.1597  lr 2.03e-06  2946s
step  1360: loss 0.1462  lr 1.97e-06  2959s
step  1370: loss 0.1781  lr 1.91e-06  2973s
step  1380: loss 0.1716  lr 1.84e-06  2987s
step  1390: loss 0.1211  lr 1.78e-06  3001s
step  1400: loss 0.1193  lr 1.71e-06  3015s
step  1410: loss 0.1713  lr 1.65e-06  3031s
step  1420: loss 0.1244  lr 1.59e-06  3043s
step  1430: loss 0.1724  lr 1.52e-06  3055s
step  1440: loss 0.1070  lr 1.46e-06  3069s
step  1450: loss 0.0976  lr 1.40e-06  3082s
step  1460: loss 0.1077  lr 1.33e-0

In [10]:
# Free the training state, reload the selected checkpoint from disk and confirm labels and outputs match what was saved.
del optimizer, scheduler, model
torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_OUT), local_files_only=True).to(device)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_OUT), local_files_only=True)
assert dict(model.config.id2label) == BASE_ID2LABEL and nli_label_ids(model.config) == ids, "label mapping changed on save"
reload_max_abs_diff = float((probe_logits(model) - saved_probe).abs().max())
assert reload_max_abs_diff < 1e-4, f"reloaded logits differ by {reload_max_abs_diff}"
checkpoint_sha256 = model_fingerprint(MODEL_OUT)
print(f"Reloaded step {best['step']} checkpoint: max logit difference {reload_max_abs_diff:.2e}; sha256 {checkpoint_sha256[:12]}")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4678.19it/s]


Reloaded step 1252 checkpoint: max logit difference 0.00e+00; sha256 a3cf162292aa


In [11]:
# The reloaded checkpoint must reproduce its selection-time validation predictions.
reloaded_validation = score(validation_examples, validation_gold)
reload_agreement = {protocol: sum(a == b for a, b in zip(reloaded_validation[protocol]["raw"], best["validation"][protocol]["raw"]))
                    / len(validation_gold) for protocol in ("per_sentence", "joined")}
assert min(reload_agreement.values()) >= 0.999, f"reloaded validation predictions differ: {reload_agreement}"
print("Validation agreement with selection time:", reload_agreement)

Validation agreement with selection time: {'per_sentence': 1.0, 'joined': 1.0}


In [12]:
# Score the held-out test claims once with the selected checkpoint (smoke: only the first test_limit claims).
n_test = config["smoke"]["test_limit"] if SMOKE_TEST else len(test_rows)
test_result = score(test_examples[:n_test], test_gold[:n_test])
print({protocol: round(test_result[protocol]["metrics"]["macro_f1"], 4) for protocol in ("per_sentence", "joined")},
      f"on {n_test:,} test claims")

{'per_sentence': 0.9561, 'joined': 0.9592} on 13,202 test claims


In [13]:
# Zero-shot references on the same test claims: the saved run for the per-sentence protocol (same code, model and ids),
# and the unchanged snapshot re-scored for the joined protocol. Each comparison stays within one protocol.
zero_shot_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR), local_files_only=True).to(device)
zero_shot_joined, _ = nli_joined_predictions(AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True),
                                             zero_shot_model, torch, test_examples[:n_test], MAX_LEN, config["eval_batch_size"])
del zero_shot_model
torch.cuda.empty_cache()
zero_shot_raw = {"per_sentence": [row["raw_prediction"] for row in zero_shot_rows[:n_test]], "joined": zero_shot_joined}
comparison = []
for protocol in ("per_sentence", "joined"):
    for model_name, raw in [("zero-shot", zero_shot_raw[protocol]), ("fine-tuned", test_result[protocol]["raw"])]:
        metrics = evaluate(test_gold[:n_test], [fever_binary_proxy(label) for label in raw], LABELS)
        comparison.append({"protocol": protocol, "role": "primary" if protocol == "per_sentence" else "secondary",
                           "model": model_name, "n": metrics["n"], "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
                           "supported_recall": metrics["per_class"]["SUPPORTED"]["recall"],
                           "not_supported_recall": metrics["per_class"]["CONTRADICTED"]["recall"],
                           "raw_unsupported": raw.count("UNSUPPORTED")})
    gold_binary = test_gold[:n_test]
    old = [fever_binary_proxy(label) == g for label, g in zip(zero_shot_raw[protocol], gold_binary)]
    new = [fever_binary_proxy(label) == g for label, g in zip(test_result[protocol]["raw"], gold_binary)]
    comparison[-1]["fixed_vs_zero_shot"] = sum(n and not o for o, n in zip(old, new))
    comparison[-1]["broken_vs_zero_shot"] = sum(o and not n for o, n in zip(old, new))
comparison = pd.DataFrame(comparison)
comparison

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3507.27it/s]


,protocol,role,model,n,accuracy,macro_f1,supported_recall,not_supported_recall,raw_unsupported,fixed_vs_zero_shot,broken_vs_zero_shot
0,per_sentence,primary,zero-shot,13202,0.814498,0.810729,0.673791,0.955034,2147,NaN,NaN
1,per_sentence,primary,fine-tuned,13202,0.956143,0.956142,0.951645,0.960636,0,2040.0,170.0
2,joined,secondary,zero-shot,13202,0.827450,0.824651,0.701531,0.953217,2062,NaN,NaN
3,joined,secondary,fine-tuned,13202,0.959173,0.959173,0.957860,0.960484,0,1898.0,159.0


In [14]:
# Save the run: settings, provenance, training log, validation history and per-protocol test predictions.
versions = {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__}
record = {"mode": MODE, "config": config, "versions": versions, "base_model_sha256": config["model_snapshot_sha256"],
          "checkpoint_dir": str(MODEL_OUT.relative_to(PROJECT_ROOT)), "checkpoint_sha256": checkpoint_sha256,
          "train_split_manifest_sha256": hashlib.sha256((split_dir / "split_manifest.json").read_bytes()).hexdigest(),
          "train_split_counts": split_manifest["split"], "truncated_share": truncated_share,
          "selected_step": best["step"], "selected_validation_macro_f1": best["macro_f1"], "total_steps_full_schedule": total_steps,
          "timing": timing, "memory": memory, "loss_log": loss_log, "validation_history": history,
          "zero_shot_validation": {p: zero_shot_validation[p]["metrics"] for p in ("per_sentence", "joined")},
          "reload": {"max_abs_logit_diff": reload_max_abs_diff, "validation_agreement": reload_agreement},
          "test_scope": f"first {n_test} test claims (smoke)" if SMOKE_TEST else f"all {n_test} test claims, scored once",
          "comparison": comparison.to_dict(orient="records"),
          "protocol_note": "Zero-shot vs fine-tuned is compared within one protocol only; per-sentence is primary."}
(RUN_DIR / "run.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
for protocol in ("per_sentence", "joined"):
    save_run(RUN_DIR / protocol, {"protocol": config[f"{'primary' if protocol == 'per_sentence' else 'secondary'}_protocol"], "mode": MODE},
             [{"claim_id": row["claim"]["claim_id"], "gold": gold, "raw_prediction": raw, "binary_prediction": binary,
               "zero_shot_raw_prediction": zero_shot}
              for row, gold, raw, binary, zero_shot in zip(test_rows, test_gold[:n_test], test_result[protocol]["raw"],
                                                           test_result[protocol]["binary"], zero_shot_raw[protocol])],
             test_result[protocol]["metrics"])
print("Saved to", RUN_DIR.relative_to(PROJECT_ROOT))

Saved to outputs\task_b\fever_finetune\run_20261022T191303Z


In [15]:
# Summary. Smoke-test numbers are engineering checks only and must not be reported or used to choose anything.
print(MODE.upper())
print(f"{timing['optimizer_steps']} optimizer steps, {timing['examples_seen']:,} examples in {timing['train_seconds']:.0f}s of training "
      f"({timing['train_examples_per_second']:.1f} examples/s, {timing['seconds_per_optimizer_step']:.2f}s per step); "
      f"evaluation and saving {timing['eval_and_save_seconds']:.0f}s; peak GPU memory {memory['peak_allocated_gib']:.2f} GiB allocated, "
      f"{memory['peak_reserved_gib']:.2f} GiB reserved")
print(f"Selected step {best['step']} by validation per-sentence macro-F1 {best['macro_f1']:.4f} "
      f"(zero-shot {zero_shot_validation['per_sentence']['metrics']['macro_f1']:.4f})")
display(pd.DataFrame(history))
display(pd.DataFrame(loss_log))

FULL TRAINING RUN
1669 optimizer steps, 53,378 examples in 2202s of training (24.2 examples/s, 1.32s per step); evaluation and saving 1576s; peak GPU memory 6.68 GiB allocated, 8.09 GiB reserved
Selected step 1252 by validation per-sentence macro-F1 0.9607 (zero-shot 0.8066)


,step,examples_seen,per_sentence_macro_f1,joined_macro_f1,per_sentence_raw_unsupported,eval_seconds,selected_so_far
0,417,13344,0.954095,0.958383,0,347.235153,417
1,834,26688,0.957124,0.962421,0,427.970906,834
2,1252,40064,0.960654,0.966204,0,392.970456,1252
3,1669,53378,0.959394,0.965448,0,393.763318,1252


,step,mean_loss,lr,elapsed_seconds
0,10,0.379081,1.000000e-06,8.253821
1,20,0.350921,2.000000e-06,16.031930
2,30,0.306691,3.000000e-06,23.886993
3,40,0.264472,4.000000e-06,31.980429
4,50,0.195871,5.000000e-06,39.774691
...,...,...,...,...
161,1620,0.122454,3.123008e-07,3321.294757
162,1630,0.158870,2.485660e-07,3335.556862
163,1640,0.130739,1.848311e-07,3349.944744
164,1650,0.200352,1.210962e-07,3362.386857
